# Preprocesameinto

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

df = pd.read_csv('data/train.csv')  # ajustar ruta según estructura del repo
df.shape

(1168, 81)

In [ ]:
TARGET = 'SalePrice'

NONE_COLS = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

QUAL_MAP = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
QUAL_COLS = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

BSMT_EXPOSURE_MAP = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
BSMT_FINTYPE_MAP = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
GARAGE_FINISH_MAP = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
FUNCTIONAL_MAP = {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4,
                   'Min2': 5, 'Min1': 6, 'Typ': 7}
LOTSHAPE_MAP = {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3}
LANDSLOPE_MAP = {'Sev': 0, 'Mod': 1, 'Gtl': 2}
PAVEDDRIVE_MAP = {'N': 0, 'P': 1, 'Y': 2}
UTILITIES_MAP = {'NoSeWa': 0, 'AllPub': 1}
CENTRALAIR_MAP = {'N': 0, 'Y': 1}

ORDINAL_MAPS = {
    'BsmtExposure': BSMT_EXPOSURE_MAP,
    'BsmtFinType1': BSMT_FINTYPE_MAP,
    'BsmtFinType2': BSMT_FINTYPE_MAP,
    'GarageFinish': GARAGE_FINISH_MAP,
    'Functional': FUNCTIONAL_MAP,
    'LotShape': LOTSHAPE_MAP,
    'LandSlope': LANDSLOPE_MAP,
    'PavedDrive': PAVEDDRIVE_MAP,
    'Utilities': UTILITIES_MAP,
    'CentralAir': CENTRALAIR_MAP,
}

LOTFRONTAGE_COL = 'LotFrontage'

ALL_ORDINAL_COLS = QUAL_COLS + list(ORDINAL_MAPS.keys())


In [3]:
class AmesFeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = X.copy()
        for col in NONE_COLS:
            if col in X.columns:
                X[col] = X[col].fillna('None')
        self.frontage_medians_ = X.groupby('Neighborhood')[LOTFRONTAGE_COL].median()
        self.frontage_global_median_ = X[LOTFRONTAGE_COL].median()
        return self

    def transform(self, X):
        X = X.copy()

        for col in NONE_COLS:
            if col in X.columns:
                X[col] = X[col].fillna('None')

        X[LOTFRONTAGE_COL] = X.apply(
            lambda row: self.frontage_medians_.get(row['Neighborhood'],
                                                     self.frontage_global_median_)
            if pd.isna(row[LOTFRONTAGE_COL]) else row[LOTFRONTAGE_COL],
            axis=1
        )

        if 'MasVnrArea' in X.columns:
            X['MasVnrArea'] = X['MasVnrArea'].fillna(0)
        if 'GarageYrBlt' in X.columns:
            X['GarageYrBlt'] = X['GarageYrBlt'].fillna(0)
        if 'Electrical' in X.columns:
            X['Electrical'] = X['Electrical'].fillna(X['Electrical'].mode()[0])

        for col in QUAL_COLS:
            if col in X.columns:
                X[col] = X[col].fillna('None').map(QUAL_MAP)

        for col, mapping in ORDINAL_MAPS.items():
            if col in X.columns:
                X[col] = X[col].fillna(list(mapping.keys())[0]).map(mapping)

        return X


## Separación en train y validation

In [4]:
X = df.drop(columns=[TARGET, 'Id'])
y = np.log1p(df[TARGET])  #transformación log 

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (934, 79), Val: (234, 79)


In [6]:
feat_eng = AmesFeatureEngineer()
X_train_fe = feat_eng.fit_transform(X_train)
X_val_fe = feat_eng.transform(X_val)

nominal_cols = X_train_fe.select_dtypes(include='object').columns.tolist()
numeric_cols = X_train_fe.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Nominales (one-hot): {len(nominal_cols)}")
print(f"Numéricas u ordinales (escaladas): {len(numeric_cols)}")

Nominales (one-hot): 23
Numéricas u ordinales (escaladas): 56


In [7]:
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), nominal_cols),
])

X_train_processed = preprocessor.fit_transform(X_train_fe)
X_val_processed = preprocessor.transform(X_val_fe)

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_val_processed shape: {X_val_processed.shape}")

X_train_processed shape: (934, 223)
X_val_processed shape: (234, 223)


# Guardar pipeline completo

In [8]:
joblib.dump({
    'feature_engineer': feat_eng,
    'preprocessor': preprocessor,
    'numeric_cols': numeric_cols,
    'nominal_cols': nominal_cols,
}, 'models/preprocessing_pipeline.joblib') 

np.save('data/X_train_processed.npy', X_train_processed)
np.save('data/X_val_processed.npy', X_val_processed)
np.save('data/y_train.npy', y_train.values)
np.save('data/y_val.npy', y_val.values)

print("Pipeline y datos procesados guardados correctamente")

Pipeline y datos procesados guardados correctamente
